# Causal Inference Analysis of Chemotherapy for Colon Cancer

### Introduction

This project aims to evaluate the **causal effect of chemotherapy treatment** on patient outcomes in stage B/C colon cancer using observational data.

The dataset contains clinical and demographic information on patients, including:
- treatment assignment (chemotherapy vs observation),
- patient characteristics (age, sex, tumor features),
- and survival outcomes (death, recurrence, follow-up time).

### Objective

The primary goal is to estimate:

> **The causal effect of chemotherapy (Lev+5FU) on the probability of death**

Since the data is observational (not randomized), treatment assignment may be influenced by patient characteristics. Therefore, we must adjust for **confounding variables** to obtain a valid causal estimate.

### Methodology

We apply a **causal inference framework** using:

- **Propensity Score Modeling** to estimate treatment assignment probability  
- **Inverse Probability of Treatment Weighting (IPTW)** to adjust for confounding  
- **Weighted Logistic Regression (GLM)** to estimate treatment effect  

This approach allows us to approximate a randomized experiment and compute:
- treatment effect (log-odds and odds ratio),
- statistical significance (p-value),
- confidence intervals.

### Key Assumptions
The causal interpretation relies on:
- **No unmeasured confounding**  
- **Correct model specification (propensity + outcome model)**  
- **Positivity (overlap between treatment groups)**  

### Outcome Definition
We focus on:
- **Death (etype = 2)** as the primary outcome 

### Dataset 

The dataset contains clinical and survival information for patients with stage B/C colon cancer, including treatment type, patient demographics, tumor characteristics, and disease outcomes. Key variables include rx (treatment group), age and sex (demographics), and several clinical factors such as nodes, obstruct, perfor, adhere, differ, extent, and surg, which describe disease severity and surgical conditions. The outcome-related variables include time (follow-up duration), status (whether the event occurred), and etype (type of event: recurrence or death). Together, these variables allow for analyzing treatment effects while accounting for patient risk factors and outcomes.  

Source of dataset: The dataset used in this analysis is publicly available and was obtained from the causal inference teaching materials provided at https://toby-codigos.github.io/ForCausality/.

### Step 1: Data Preparation

Load the dataset, keep only death outcomes (`etype = 2`), and select two treatment groups (Obs vs Lev+5FU).  
Create binary variables for treatment (`T`) and outcome (`Y`), and remove missing values.

In [ ]:
# Step 1: Load and prepare data
import pandas as pd
import numpy as np

df = pd.read_csv("colon_cancer_chemotherapy.csv")

# keep only death outcome
df = df[df["etype"] == 2].copy()

# keep only 2 treatment groups
df = df[df["rx"].isin(["Obs", "Lev+5FU"])]

# treatment
df["T"] = (df["rx"] == "Lev+5FU").astype(int)

# outcome
df["Y"] = df["status"]

# drop missing
df = df.dropna()



### Step 2: Confounder Selection

Define a set of baseline variables (demographic and clinical factors) that may influence both treatment assignment and the outcome.

In [8]:
confounders = [
    "age", "sex",
    "nodes", "node4",
    "obstruct", "perfor", "adhere",
    "differ", "extent",
    "surg"
]

### Step 3: Estimate Propensity Scores (statsmodels)

Estimate the propensity score by fitting a logistic regression model that predicts treatment assignment from the confounding variables, then compute each patient’s predicted probability of receiving the treatment.

In [10]:
import statsmodels.api as sm

X_ps = df[confounders]
X_ps = sm.add_constant(X_ps)

ps_model = sm.Logit(df["T"], X_ps).fit()
print(ps_model.summary())

# predicted propensity score
df["ps"] = ps_model.predict(X_ps)

Optimization terminated successfully.
         Current function value: 0.686317
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:                      T   No. Observations:                  594
Model:                          Logit   Df Residuals:                      583
Method:                           MLE   Df Model:                           10
Date:                Fri, 05 Jun 2026   Pseudo R-squ.:                0.009335
Time:                        22:53:36   Log-Likelihood:                -407.67
converged:                       True   LL-Null:                       -411.51
Covariance Type:            nonrobust   LLR p-value:                    0.6598
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.2332      0.723      0.323      0.747      -1.184       1.650
age            0.0022      0.

None of the confounders show statistically significant effects on treatment assignment (all p-values > 0.05), suggesting that treatment assignment is not strongly predicted by these observed variables.

### Step 4: Compute IPTW weights

Compute inverse probability weights by assigning each treated unit a weight of 1/ps and each control unit a weight of 1/(1-ps) based on their propensity score.


In [11]:
df["weight"] = np.where(
    df["T"] == 1,
    1 / df["ps"],
    1 / (1 - df["ps"])
)

### Step 5: Weighted Logistic Regression (CAUSAL MODEL)

Fit a weighted logistic regression model using IPTW to estimate the causal effect of treatment on the probability of death and summarize the results.

In [12]:
X_out = sm.add_constant(df["T"])  # only treatment in outcome model

model = sm.GLM(
    df["Y"],
    X_out,
    family=sm.families.Binomial(),
    freq_weights=df["weight"]
)

result = model.fit()

print(result.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                      Y   No. Observations:                  594
Model:                            GLM   Df Residuals:                  1185.76
Model Family:                Binomial   Df Model:                            1
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -813.23
Date:                Fri, 05 Jun 2026   Deviance:                       1626.5
Time:                        22:54:30   Pearson chi2:                 1.19e+03
No. Iterations:                     4   Pseudo R-squ. (CS):            0.02477
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0904      0.082      1.101      0.2

### Causal Effect of Chemotherapy on Death (IPTW + Logistic Regression)

### Model Summary

We estimated the causal effect of chemotherapy (`T`) on the probability of death (`Y`) using:

- Inverse Probability Weighting (IPTW)
- Weighted logistic regression (GLM with logit link)

###  Key Result

### Treatment Effect (`T`)

- Coefficient (log-odds): **-0.4504**
- Standard Error: 0.117
- z-value: -3.847
- p-value: **< 0.001**
- 95% CI: [-0.680, -0.221]

###  Statistical Significance

- The p-value is **< 0.001**, which means:
  
   The effect of chemotherapy on death is **statistically significant**

###  Direction of Effect

- The coefficient is **negative (-0.4504)**

   Chemotherapy **reduces the risk of death**


###  Convert to Odds Ratio

Odds Ratio = exp(-0.4504), which is approximately 0.64.

 Interpretation:

> Patients receiving chemotherapy (**Lev+5FU**) have about **36% lower odds of death** compared to the observation group, after adjusting for confounding.

###  Confidence Interval

exp(-0.680) is approximately 0.51, and exp(-0.221) is approximately 0.80.

 Interpretation:

> The true effect likely reduces death odds by **20% to 49%**, and this effect is consistently protective.


###  Constant (Intercept)

- Coefficient: 0.0904
- p-value: 0.271

 Interpretation:

- Not statistically significant
- Represents baseline log-odds of death when `T = 0`
- Not of primary interest


###  Final Conclusion

> **Chemotherapy (Lev+5FU) significantly reduces the probability of death** in patients with colon cancer.

- Effect is statistically significant (**p < 0.001**)
- Effect size is meaningful (≈ **36% reduction in odds of death**)


###  Important Notes

- This is a **causal estimate under the assumption of no unmeasured confounding**
- Censoring is treated as "alive" (simplification)
- Results should be interpreted cautiously but are **strong evidence of treatment benefit**